# Diabetes Prediction Simulation Experiments

**Goal:**
- Study how structural inequalities in features (SES, BMI, labs) can lead to disparities in model predictions across groups (race/gender).
- Compare how different models (Logistic Regression vs Neural Network) react to these inequalities.
- Compute fairness metrics on model predictions.

**Key Notes:**
- Target: `diabetes`
- Features: age, SES, BMI, glucose, hba1c
- Sensitive attributes: race, gender



In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
def simulate_population(n_samples=10000, seed=42):
    np.random.seed(seed)
    race = np.random.choice([0,1,2], n_samples, p=[0.6,0.25,0.15])
    gender = np.random.choice([0,1], n_samples)
    age = np.random.normal(50, 12, n_samples)
    ses = 0.5*(race==0) + 0.0*(race==1) - 0.5*(race==2) + np.random.normal(0, 0.3, n_samples)
    bmi = 27 + 1.0*(gender==1) - 1.2*ses + np.random.normal(0, 2, n_samples)
    glucose = 95 + 2*(bmi-27) + np.random.normal(0, 6, n_samples)
    hba1c = 5.4 + 0.1*(bmi-27) + np.random.normal(0, 0.2, n_samples)
    risk = 0.1*(glucose-100) + 1.0*(hba1c-5.4) + 0.5*(bmi-27) + np.random.normal(0, 0.2, n_samples)
    diabetes_true = np.random.binomial(1, 1/(1+np.exp(-risk)))
    df = pd.DataFrame({'race': race, 'gender': gender, 'age': age, 'ses': ses, 'bmi': bmi, 'glucose': glucose, 'hba1c': hba1c, 'diabetes_true': diabetes_true})
    return df

In [25]:
def add_measurement_bias(df, seed=42):
    np.random.seed(seed)

    df = df.copy()

    mask_r2 = df['race'] == 2

    # Add extra noise to measurements for race group 2
    df.loc[mask_r2, 'glucose'] += np.random.normal(0, 12, mask_r2.sum())
    df.loc[mask_r2, 'hba1c']  += np.random.normal(0, 0.4, mask_r2.sum())

    return df


In [26]:
def train_model_true(df, test_size=0.3, seed=42):
    features = ['age', 'ses', 'bmi', 'glucose', 'hba1c']
    X = df[features]
    y = df['diabetes_true']

    X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
        X, y, df, test_size=test_size, stratify=df['race'], random_state=seed
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_scaled, y_train)

    df_test['pred'] = model.predict(X_test_scaled)

    return model, df_train, df_test, scaler


In [27]:
def compute_EO(df, group_col, label_col='diabetes_true', pred_col='pred'):
    results = []
    for g in np.unique(df[group_col]):
        mask = df[group_col]==g
        tpr = ((df[pred_col][mask]==1) & (df[label_col][mask]==1)).sum() / (df[label_col][mask]==1).sum()
        fpr = ((df[pred_col][mask]==1) & (df[label_col][mask]==0)).sum() / (df[label_col][mask]==0).sum()
        results.append({'group': g, 'TPR': tpr, 'FPR': fpr})
    return pd.DataFrame(results)

In [28]:
df = simulate_population()
df = add_measurement_bias(df)

model, df_train, df_test, scaler = train_model_true(df)

race_metrics = compute_EO(df_test, 'race')
gender_metrics = compute_EO(df_test, 'gender')

print("EO Metrics by Race")
print(race_metrics)

print("EO Metrics by Gender")
print(gender_metrics)


EO Metrics by Race
   group       TPR       FPR
0      0  0.626490  0.161560
1      1  0.770701  0.217184
2      2  0.778243  0.311224
EO Metrics by Gender
   group       TPR       FPR
0      0  0.626571  0.137421
1      1  0.735020  0.262735


In [29]:
from graphviz import Digraph

# Create a directed graph
dot = Digraph(comment='Diabetes Graphical Model')

# Nodes
nodes = ['race', 'gender', 'age', 'ses', 'bmi', 'glucose', 'hba1c', 'risk', 'diabetes_true']
for node in nodes:
    dot.node(node, node)

# Edges - showing causal dependencies based on your simulation
edges = [
    ('race', 'ses'),          # SES depends on race
    ('gender', 'bmi'),        # BMI depends on gender
    ('ses', 'bmi'),           # BMI depends on SES
    ('bmi', 'glucose'),       # glucose depends on BMI
    ('bmi', 'hba1c'),         # hba1c depends on BMI
    ('glucose', 'risk'),      # risk depends on glucose
    ('hba1c', 'risk'),        # risk depends on hba1c
    ('bmi', 'risk'),          # risk depends on BMI
    ('risk', 'diabetes_true') # diabetes depends on risk
]
for start, end in edges:
    dot.edge(start, end)

# Render the graph
print(dot.source)
dot.render('diabetes_graphical_model', view=True)

// Diabetes Graphical Model
digraph {
	race [label=race]
	gender [label=gender]
	age [label=age]
	ses [label=ses]
	bmi [label=bmi]
	glucose [label=glucose]
	hba1c [label=hba1c]
	risk [label=risk]
	diabetes_true [label=diabetes_true]
	race -> ses
	gender -> bmi
	ses -> bmi
	bmi -> glucose
	bmi -> hba1c
	glucose -> risk
	hba1c -> risk
	bmi -> risk
	risk -> diabetes_true
}



'diabetes_graphical_model.pdf'